# Confidently Wrong - smoke test

*Calibration Risks and Safe Abstention in African-Language LLMs. Apart Global South AI Safety Hackathon 2026 (Africa Track).*

Two jobs: (1) preview every report figure on dummy data so we can lock the plot style before
spending GPU time, and (2) run the FULL pipeline (run -> analyze -> compare) end to end on
Qwen3-4B with a few samples across both datasets, plus load checks for the open-weight roster.

Run with:
```
uv run jupyter nbconvert --to notebook --execute --inplace notebooks/smoke_test.ipynb
```
For a fast CPU plumbing run (no big downloads), set
`SMOKE_MODEL_ID=hf-internal-testing/tiny-random-LlamaForCausalLM`.

In [ ]:
import os, sys, gc, subprocess
from pathlib import Path
from collections import Counter

REPO_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from omegaconf import OmegaConf
from src.utils.io import load_dotenv
from src.utils.plotting import set_style

load_dotenv(REPO_ROOT / ".env")
set_style()
HF_TOKEN = os.environ.get("HF_TOKEN")
MODEL_CFG = os.environ.get("SMOKE_MODEL_CFG", "qwen3_4b_instruct")
TINY = os.environ.get("SMOKE_MODEL_ID")
print("repo:", REPO_ROOT, "| HF_TOKEN set:", bool(HF_TOKEN), "| tiny override:", bool(TINY))


def free():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def patch_tiny(mcfg):
    if TINY:
        mcfg.hf_id = TINY
        mcfg.device_map = "cpu"
        mcfg.attn_implementation = "eager"
    return mcfg

## Report-figure preview (dummy data, no model or download required)

Synthesizes the expected story - accuracy decays and confidence does NOT, so calibration
error grows as languages get lower-resource - and renders every report figure through the real
`src` plotting code. Run this first to settle the plot style; nothing here touches a model.

In [ ]:
from src.analysis.metrics_tables import per_language_metrics, confidence_correct
from src.analysis.figures import (save_reliability_grid, save_risk_coverage,
                                  save_language_bars, save_language_radar,
                                  save_topic_calibration_radar)
from src.metrics.selective import build_deployment_card
from src.utils.io import experiment_paths
from IPython.display import Image, display

DUMMY = [("eng", 0.85, 0.03), ("swa", 0.62, 0.10), ("yor", 0.48, 0.20),
         ("hau", 0.42, 0.24), ("amh", 0.33, 0.30), ("nso", 0.30, 0.33)]


def synth_lang(lang, acc, overconf, n=300, seed=0):
    rng = np.random.default_rng(seed)
    q = np.clip(rng.normal(acc, 0.15, n), 0.02, 0.98)
    correct = (rng.uniform(size=n) < q).astype(int)
    conf = np.clip(q + overconf + rng.normal(0, 0.05, n), 0.05, 0.999)
    return [
        {"language": lang, "confidence": float(c), "correct": bool(y), "n_choices": 4,
         "domain": None, "verbalized_confidence": None, "predicted_index": 0,
         "answer_index": 0, "scoring": "mc1", "mode": "mc1"}
        for c, y in zip(conf, correct)
    ]


dummy_preds = []
for i, (lang, acc, oc) in enumerate(DUMMY):
    dummy_preds += synth_lang(lang, acc, oc, seed=i)

prev = experiment_paths(REPO_ROOT / "artifacts" / "_preview")
lang_df = per_language_metrics(dummy_preds, n_bins=15, bootstrap_n=0)
print(lang_df.to_string(index=False))

LABEL = "Uhura-TruthfulQA (dummy)"
save_reliability_grid(dummy_preds, prev["figures"], n_bins=15, title_label=LABEL)   # Act 2
save_risk_coverage(dummy_preds, prev["figures"], title_label=LABEL)                 # Act 3
save_language_bars(lang_df, prev["figures"], title_label=LABEL)                      # Act 1
save_language_radar(lang_df, prev["figures"], title_label=LABEL)                     # radar

per_lang = {lang: confidence_correct(dummy_preds, language=lang) for lang, _, _ in DUMMY}
card = pd.DataFrame(build_deployment_card(per_lang, [0.5, 0.7, 0.9]))
card.to_csv(prev["results"] / "deployment_card.csv", index=False)
print("\nDeployment card (dummy):\n", card.round(3).to_string(index=False))

for png in ["reliability_by_language.png", "risk_coverage.png",
            "accuracy_by_language.png", "ece_by_language.png", "language_radar.png"]:
    display(Image(str(prev["figures"] / png)))

### Topic-calibration radar (Uhura): which topics is the model confidently wrong about?

Joins each Uhura question to its TruthfulQA topic (from the `_generation` config) and shows
calibration (1 - ECE) by topic, English vs African languages. Here on dummy data; the real
run produces it for Uhura automatically.

In [ ]:
TOPICS = ["Health", "Law", "Finance", "Politics", "Misconceptions",
          "Economics", "Paranormal", "Sociology"]
HARD = {"Health", "Law", "Finance", "Economics", "Misconceptions"}
rng2 = np.random.default_rng(3)
topic_preds = []
for lang, is_eng in [("eng", True), ("yor", False), ("swa", False), ("hau", False)]:
    for t in TOPICS:
        for _ in range(40):
            acc = 0.72 if is_eng else (0.40 if t in HARD else 0.52)
            oc = 0.04 if is_eng else (0.32 if t in HARD else 0.16)
            q = float(np.clip(rng2.normal(acc, 0.15), 0.02, 0.98))
            topic_preds.append({"language": lang, "domain": t,
                                "confidence": float(np.clip(q + oc + rng2.normal(0, 0.05), 0.05, 0.999)),
                                "correct": bool(rng2.uniform() < q),
                                "verbalized_confidence": None, "n_choices": 4})
save_topic_calibration_radar(topic_preds, prev["figures"], title_label="Uhura-TruthfulQA (dummy)")
display(Image(str(prev["figures"] / "topic_calibration_radar.png")))

### Cross-model figures (dummy): heatmaps, decay lines, radar, reasoning vs non-reasoning

These mirror `compare.py` once several models have run (the time-permitting extension), plus
the E6 reasoning-vs-non-reasoning calibration comparison.

In [ ]:
from src.utils.plotting import PALETTE, heatmap, radar, set_style

set_style()
models = ["qwen3_4b_instruct", "qwen3_4b_thinking", "gemma4_12b", "afrollama_v1"]
langs = [d[0] for d in DUMMY]
rng = np.random.default_rng(7)
acc_mat = np.clip(np.array([[a * (0.8 + 0.07 * m) + rng.normal(0, 0.015) for _, a, _ in DUMMY]
                            for m in range(len(models))]), 0, 1)
ece_mat = np.clip(np.array([[oc * (1.0 - 0.04 * m) + abs(rng.normal(0, 0.01)) for _, _, oc in DUMMY]
                            for m in range(len(models))]), 0, 1)

fig, ax = plt.subplots(figsize=(8, 3.0), constrained_layout=True)
heatmap(acc_mat, models, langs, ax=ax, title="Accuracy (model x language) [dummy]", cbar_label="accuracy")
fig.savefig(prev["figures"] / "accuracy_heatmap.png"); plt.show()

fig, ax = plt.subplots(figsize=(8, 3.0), constrained_layout=True)
heatmap(ece_mat, models, langs, ax=ax, title="ECE (model x language) [dummy]", cbar_label="ECE", fmt="{:.3f}")
fig.savefig(prev["figures"] / "ece_heatmap.png"); plt.show()

fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
for i, m in enumerate(models):
    ax.plot(langs, acc_mat[i], marker="o", lw=1.7, color=PALETTE[i % len(PALETTE)], label=m)
ax.set_title("Factuality decay across languages [dummy]"); ax.set_ylabel("accuracy")
ax.set_xlabel("language (high to low resource)"); ax.tick_params(axis="x", rotation=45)
ax.legend(fontsize=7); fig.savefig(prev["figures"] / "accuracy_decay.png"); plt.show()

series = {m: acc_mat[i].tolist() for i, m in enumerate(models)}
fig, ax = plt.subplots(figsize=(6.8, 6.8), subplot_kw={"polar": True}, constrained_layout=True)
radar(langs, series, ax=ax, title="Accuracy across languages by model [dummy]")
fig.savefig(prev["figures"] / "accuracy_radar.png"); plt.show()

x = np.arange(len(langs)); w = 0.38
fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
ax.bar(x - w / 2, ece_mat[0], w, color=PALETTE[4], label="non-thinking")
ax.bar(x + w / 2, ece_mat[0] * 1.18, w, color=PALETTE[0], label="thinking")
ax.set_xticks(x); ax.set_xticklabels(langs, rotation=45, ha="right"); ax.set_ylabel("ECE")
ax.set_title("Reasoning vs non-reasoning calibration [dummy]"); ax.legend(fontsize=8)
fig.savefig(prev["figures"] / "reasoning_vs_nonreasoning_ece.png"); plt.show()
print("preview figures written to", prev["figures"])

## Step 0 - verify the data before any GPU run (most important)

List the actual HF *config* names for each dataset. If your `configs/dataset/*.yaml` language
codes do not appear here, fix them: a wrong code silently fails the loader, and a silent field
drop invalidates a whole run.

In [ ]:
from datasets import get_dataset_config_names

for hf_id in ["masakhane/afrimmlu", "masakhane/uhura-truthfulqa"]:
    try:
        cfgs = get_dataset_config_names(hf_id, token=HF_TOKEN)
        print(hf_id, "->", cfgs)
    except Exception as e:
        print("FAILED to list configs for", hf_id, ":", e)

## Step 1 - load two languages per dataset, print one full prompt verbatim

AfriMMLU config names are the language codes; Uhura uses a `<code>_multiple_choice` config
(mapped via `hf_config_map`) and the topic is joined from `<code>_generation`.

In [ ]:
from src.data.loader import load_samples, print_one

afri_cfg = OmegaConf.load(REPO_ROOT / "configs/dataset/afrimmlu.yaml")
uh_cfg = OmegaConf.load(REPO_ROOT / "configs/dataset/uhura_truthfulqa.yaml")

afri = load_samples(afri_cfg, languages=["eng", "yor"], token=HF_TOKEN, limit=20)
uh = load_samples(uh_cfg, languages=["eng", "yor"], token=HF_TOKEN, limit=20)

print_one(afri, "yor")
print_one(uh, "yor")

assert all(len(s.choices) == 4 for s in afri), "AfriMMLU items must have exactly 4 options"
print("AfriMMLU OK:", len(afri), "items")
print("Uhura scoring:", uh[0].scoring, "| candidate-count distribution:", Counter(len(s.choices) for s in uh))
print("Uhura topics present:", Counter(s.domain for s in uh).most_common(6))
assert all(0 <= s.answer_index < len(s.choices) for s in uh), "bad answer index in Uhura"
print("Uhura OK:", len(uh), "items")

afri_prompt = (REPO_ROOT / "prompts/afrimmlu_fixed_option.txt").read_text()
mc1_prompt = (REPO_ROOT / "prompts/uhura_truthfulqa_mc1.txt").read_text()
thinking = (REPO_ROOT / "prompts/thinking_verbalized.txt").read_text()

## Step 2 - model load checks (full open-weight roster)

Confirm each model downloads, is ungated for your account, and runs. Heavy models are freed
between checks. A FAIL here (gated / missing HF id) is exactly what we want to catch before a run.
Qwen3-4B (instruct + thinking) is the focus; Gemma 4, AfroLlama, Aya are the time-permitting extensions.

In [ ]:
from src.models.inference import ModelRunner

ROSTER = ["qwen3_4b_instruct", "qwen3_4b_thinking", "gemma4_12b", "afrollama_v1", "aya_expanse_8b"]
if TINY:
    ROSTER = ["qwen3_4b_instruct"]

probe = [s for s in afri if s.language == "yor"][:2]
for cfg_name in ROSTER:
    mcfg = patch_tiny(OmegaConf.load(REPO_ROOT / f"configs/model/{cfg_name}.yaml"))
    if mcfg.reasoning:           # keep the load-check cheap for the thinking model
        mcfg.n_samples = 2
        mcfg.max_new_tokens = 64
    try:
        runner = ModelRunner(mcfg, token=HF_TOKEN)
        preds = runner.run(probe, afri_prompt, thinking, batch_size=2)
        print("OK   %-18s device=%s  example_conf=%.3f" % (cfg_name, runner.device, preds[0].confidence))
        del runner, preds
        free()
    except Exception as e:
        print("FAIL %-18s %s" % (cfg_name, repr(e)[:200]))

## Step 3 - confidence modes on Qwen3-4B (Uhura answer-text MC1 + optional self-consistency)

In [ ]:
qcfg = patch_tiny(OmegaConf.load(REPO_ROOT / f"configs/model/{MODEL_CFG}.yaml"))
qrun = ModelRunner(qcfg, token=HF_TOKEN)
sub_uh = [s for s in uh if s.language == "eng"][:6] + [s for s in uh if s.language == "yor"][:6]
preds_uh = qrun.run(sub_uh, mc1_prompt, thinking, batch_size=1)
p = preds_uh[0]
print("uhura MC1 (answer-text) probs_sum=%.3f (~1)  n_choices=%d  confidence=%.3f" % (sum(p.probs), p.n_choices, p.confidence))
assert abs(sum(p.probs) - 1.0) < 1e-4
del qrun, preds_uh
free()

if os.environ.get("SMOKE_THINKING", "0") == "1":
    tcfg = OmegaConf.load(REPO_ROOT / "configs/model/qwen3_4b_thinking.yaml")
    tcfg.n_samples = 3
    tcfg.max_new_tokens = 512
    trun = ModelRunner(patch_tiny(tcfg), token=HF_TOKEN)
    tp = trun.run(sub_uh[:2], mc1_prompt, thinking, batch_size=1)
    for x in tp:
        print("thinking pred=%s conf=%.2f verbalized=%s" % (x.predicted_index, x.confidence, x.verbalized_confidence))
    del trun
    free()
else:
    print("set SMOKE_THINKING=1 to exercise the reasoning self-consistency path (heavy).")

## Step 4 - FULL pipeline end to end (run -> analyze -> compare)

The exact entrypoints the full experiments use, on Qwen3-4B with a few samples, across BOTH
datasets (AfriMMLU and Uhura). Each runs as a subprocess, like a real job.

In [ ]:
HW = os.environ.get("SMOKE_HARDWARE", "cpu")
overrides = []
if TINY:
    overrides = [f"model.hf_id={TINY}", "model.device_map=cpu", "model.attn_implementation=eager"]


def run_cmd(args):
    print(">>", "python -m", " ".join(args))
    r = subprocess.run([sys.executable, "-m", *args], cwd=str(REPO_ROOT), capture_output=True, text=True)
    print(r.stdout[-1500:])
    if r.returncode != 0:
        print("STDERR (tail):\n", r.stderr[-2000:])
    assert r.returncode == 0, f"command failed: {args}"


for ds in ["uhura_truthfulqa", "afrimmlu"]:
    run_cmd(["src.run", f"model={MODEL_CFG}", f"dataset={ds}", f"hardware={HW}",
             "languages=[eng,yor]", "+limit=8"] + overrides)
    run_cmd(["src.analyze", "--experiment", f"{MODEL_CFG}_{ds}", "--bootstrap", "100"])

run_cmd(["src.compare"])
print("pipeline complete.")

## Step 5 - inspect the produced artifacts (this is what the full run yields, scaled up)

In [ ]:
from IPython.display import Image, display

for ds in ["uhura_truthfulqa", "afrimmlu"]:
    exp = f"{MODEL_CFG}_{ds}"
    art = REPO_ROOT / "artifacts" / exp
    need = ["results/predictions.jsonl", "results/metrics_by_language.csv",
            "results/deployment_card.csv", "figures/reliability_by_language.png",
            "figures/risk_coverage.png", "figures/language_radar.png"]
    missing = [f for f in need if not (art / f).exists()]
    assert not missing, f"{exp} missing artifacts: {missing}"
    print("===", exp, "===")
    print(pd.read_csv(art / "results/metrics_by_language.csv").to_string(index=False))
    print()

cmp_dir = REPO_ROOT / "artifacts" / "_comparison"
assert (cmp_dir / "all_metrics.csv").exists(), "compare did not write outputs"
print("artifacts OK for both datasets + comparison. smoke test complete.")
display(Image(str(REPO_ROOT / "artifacts" / f"{MODEL_CFG}_uhura_truthfulqa" / "figures" / "reliability_by_language.png")))